In [1]:
import dashscope
import json
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import pandas as pd

# 1. 重新把模型和数据准备好（为了让这个notebook能独立跑）
dashscope.api_key = ""

df = pd.read_csv("pima-indians-diabetes.csv")
X = df.iloc[:, :-1]
y = df.iloc[:, -1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# 2. 取一个测试样本，算出患病概率
patient = X_test.iloc[0].to_dict()
risk_prob = model.predict_proba(X_test_scaled[0:1])[0, 1]

# 3. 构造一个要求输出 JSON 的 Prompt
prompt = f"""
你是一名内分泌临床医师。请根据患者指标和模型预测概率，输出一段 JSON 格式的评估结果。
患者指标：{patient}
模型预测患病概率：{risk_prob:.2%}

请严格按照以下 JSON 结构输出，不要包含任何其他文字（如 Markdown 代码块标记）：
{{
    "risk_level": "高风险/中风险/低风险",
    "main_factors": ["因素1", "因素2"],
    "suggestions": ["建议1", "建议2"],
    "disclaimer": "本结果仅供参考，不能替代临床就诊。"
}}
"""

# 4. 调用 API 并解析 JSON
resp = dashscope.Generation.call(
    model="qwen-max",
    messages=[{"role": "user", "content": prompt}]
)

if resp.status_code == 200:
    json_text = resp.output.text
    print("大模型原始输出：")
    print(json_text)
    print("-" * 30)
    
    try:
        result = json.loads(json_text)  # 把字符串变成 Python 字典
        print("解析成功！")
        print("风险等级：", result["risk_level"])
        print("主要因素：", result["main_factors"])
        print("生活建议：", result["suggestions"])
        print("免责声明：", result["disclaimer"])
    except json.JSONDecodeError:
        print("解析失败！模型没有按要求输出纯 JSON，请调整 Prompt。")
else:
    print("调用失败：", resp.message)

大模型原始输出：
{
    "risk_level": "中风险",
    "main_factors": ["血糖水平", "体重指数"],
    "suggestions": ["定期监测血糖变化", "保持健康饮食与适量运动"],
    "disclaimer": "本结果仅供参考，不能替代临床就诊。"
}
------------------------------
解析成功！
风险等级： 中风险
主要因素： ['血糖水平', '体重指数']
生活建议： ['定期监测血糖变化', '保持健康饮食与适量运动']
免责声明： 本结果仅供参考，不能替代临床就诊。


In [2]:
import dashscope
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ========== 1. 准备模型和数据 ==========
dashscope.api_key = ""  # 换成你自己的

df = pd.read_csv("pima-indians-diabetes.csv")
X = df.iloc[:, :-1]
y = df.iloc[:, -1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# 取第一个测试样本
patient = X_test.iloc[0].to_dict()
risk_prob = model.predict_proba(X_test_scaled[0:1])[0, 1]

# ========== 2. 定义调用 Qwen 的函数 ==========
def call_qwen(messages):
    """封装 API 调用，传入 messages 列表，返回响应文本"""
    resp = dashscope.Generation.call(
        model="qwen-max",
        messages=messages
    )
    if resp.status_code == 200:
        return resp.output.text
    else:
        return f"调用失败：{resp.message}"

# ========== 3. 第一轮：发送指标，获取 JSON 报告 ==========
# 构造第一轮对话的 messages 列表
messages = [
    {
        "role": "user",
        "content": f"""
你是一名内分泌临床医师。请根据患者指标和模型预测概率，输出一段 JSON 格式的评估结果。
患者指标：{patient}
模型预测患病概率：{risk_prob:.2%}

请严格按照以下 JSON 结构输出，不要包含任何其他文字：
{{
    "risk_level": "高风险/中风险/低风险",
    "main_factors": ["因素1", "因素2"],
    "suggestions": ["建议1", "建议2"],
    "disclaimer": "本结果仅供参考，不能替代临床就诊。"
}}
"""
    }
]

# 第一次调用
first_reply = call_qwen(messages)
print("【第一轮 · 机器读取的结构化报告】")
print(first_reply)
print("=" * 50)

# 把 AI 的回答加入对话历史（关键！）
messages.append({"role": "assistant", "content": first_reply})

# ========== 4. 第二轮：患者追问 ==========
follow_up = "我平时特别爱吃米饭和甜食，听了您的建议后，我具体应该怎么调整饮食？"
messages.append({"role": "user", "content": follow_up})

# 第二次调用（带上了前面的历史记录）
second_reply = call_qwen(messages)
print("【第二轮 · 患者追问后，模型的回答】")
print(second_reply)
print("=" * 50)

# ========== 5. 解析第一轮 JSON 并给人类看 ==========
try:
    result = json.loads(first_reply)
    print("\n===== 给人类看的最终报告 =====")
    print(f"风险等级：{result['risk_level']}")
    print(f"主要因素：{'、'.join(result['main_factors'])}")
    print("生活建议：")
    for i, s in enumerate(result["suggestions"], 1):
        print(f"  {i}. {s}")
    print(f"免责声明：{result['disclaimer']}")
except json.JSONDecodeError:
    print("解析失败！模型没有输出纯 JSON，请检查 Prompt。")

【第一轮 · 机器读取的结构化报告】
{
    "risk_level": "中风险",
    "main_factors": ["血糖水平", "年龄"],
    "suggestions": ["定期监测血糖变化", "增加体育活动，保持健康体重"],
    "disclaimer": "本结果仅供参考，不能替代临床就诊。"
}
【第二轮 · 患者追问后，模型的回答】
了解您的饮食习惯后，以下是一些建议帮助您调整饮食结构，以促进血糖控制和整体健康：

1. **减少精制碳水化合物的摄入**：尝试减少米饭等精制谷物的分量，并用全谷类（如糙米、燕麦）代替部分精白米。全谷类富含膳食纤维，有助于延缓餐后血糖上升速度。
   
2. **控制甜食摄入**：尽量限制含糖饮料、糖果以及高糖零食的消费。可以选择新鲜水果作为健康的甜品替代选项，但注意适量，因为某些水果也含有较高的天然糖分。

3. **均衡膳食**：确保每顿饭都包含足够的蔬菜（非淀粉性），适量蛋白质来源（如鱼肉、鸡胸肉、豆制品等）及少量优质脂肪（比如橄榄油、坚果）。这样的搭配可以帮助改善胰岛素敏感度并维持长时间饱腹感。

4. **定时定量进餐**：保持规律的用餐时间，避免过饥或暴饮暴食。分散食物摄入至一天中的多个小餐，可以有助于平稳血糖水平。

5. **咨询专业人士**：考虑寻求营养师的帮助来制定个性化饮食计划。他们可以根据您的具体情况提供更加具体且实用的指导建议。

请记住，改变饮食习惯需要时间和耐心，逐步做出调整往往比突然大幅度变动更容易坚持下去。希望这些建议对您有所帮助！

===== 给人类看的最终报告 =====
风险等级：中风险
主要因素：血糖水平、年龄
生活建议：
  1. 定期监测血糖变化
  2. 增加体育活动，保持健康体重
免责声明：本结果仅供参考，不能替代临床就诊。
